# pecos: the pipeline, step by step

Team **pecos**, More than a Hack 2027, Phase 1 (FC Barcelona vs Athletic). This notebook builds the submitted labels
one idea at a time, in the order they were added, and ends by **writing the submission file**: the same file as
`python scripts/make_submission.py --name pecos`.

Each section adds one idea and shows two kinds of numbers:
- **public check**: spellF1 (with its 5–95% bootstrap range), macroF1 and the number of possessions, on the
  organisers' public 5 minutes. It is a check, not a target: the rules forbid tuning on it, and 17 possessions is a
  small sample;
- **full match** (no ground truth needed): possessions, per minute, how many are shorter than 2 s (fake switches),
  changes of team and the share of dead ball.

| section | idea |
|---|---|
| 0 | setup and the data |
| 1 | the organisers' baselines (reference) |
| 2 | ball lost → keep the last team |
| 3 | dead ball when the players slow down |
| 4 | dead ball when the ball leaves the pitch |
| 5 | dead ball just before a set piece |
| 6 | no votes while the ball is in flight |
| 7 | anti-flicker: short suspicious possessions |
| 8 | the referee's whistle |
| 9 | finishing touches, one at a time and together (the submitted model) |
| 10 | every step in one table |
| 11 | **the submission file** |

Runs in 1–2 minutes once the caches in `data/cache/` exist (the first run of the pipeline builds them). Nothing here
prints match data, only counts and scores.

## 0. Setup and the data

In [ ]:
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display
from games import GAME, half_video_blocks
from evaluation import possession_metrics as pm
from analysis import best, passes as ps, v9, whistle as wh
from analysis.check import check
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 80)

FPS = 25
HALVES = [(a, z + 1) for _, a, z in half_video_blocks(GAME)]
MINUTES = sum(b - a for a, b in HALVES) / FPS / 60

B = best.Bundle.load()          # tracking, tags, audio-based whistle candidates, every shared signal
ctx = B.ctx
g = ctx.graded
filled = lambda z: np.array(pm.LABELS)[pm.fill_abstentions(pm.label_codes(np.asarray(z)), HALVES)]

codes = pd.Series([e["code"].upper() for e in ctx.tags]).value_counts()
print(f"{ctx.N:,} frames; {g.sum():,} graded ({MINUTES:.1f} min over the two halves)")
print(f"ball tracked on {ctx.ok[g].mean():.1%} of graded frames, missing on {(~ctx.ok[g]).mean():.1%}")
print(f"{len(ctx.tags)} Smart Tagging tags in {len(codes)} kinds; SET PIECES {codes.get('SET PIECES', 0)}, LONG BALL {codes.get('LONG BALL', 0)}")

In [ ]:
STEPS = {}

def full_match(z):
    f = filled(z)
    spells = pm.possession_spells(pm.label_codes(f), HALVES)
    lengths = np.array([b - a for a, b, _ in spells]) / FPS
    changes = 0
    for a0, b0 in HALVES:
        teams = [k for a, b, k in spells if a >= a0 and b <= b0]
        changes += sum(x != y for x, y in zip(teams, teams[1:]))
    return {"possessions": len(spells), "per minute": round(len(spells) / MINUTES, 2),
            "shorter than 2 s": int((lengths < 2).sum()), "changes of team": changes,
            "dead-ball share": round(float(np.mean(f[g] == "D")), 3)}

def show(name, labels, keep=True):
    "score one model: public check + full match; keep=True adds it to the summary"
    if keep:
        STEPS[name] = labels
    c = check(labels, name, log=False).iloc[0]
    row = {"spellF1": c.spellF1, "range": f"{c.spell_lo:.2f}-{c.spell_hi:.2f}", "macroF1": c.macroF1,
           "poss (truth 17)": c.poss, **full_match(labels)}
    return pd.DataFrame([row], index=[name])

## 1. The organisers' baselines (reference)

Each frame, the team of the player nearest the ball, then a **majority vote over ±N frames** (`starter/baseline.py`,
`smooth`). Frames with no tracked ball become dead ball. `smooth50` is ±2 s; `smooth125` (±5 s) is their best on the
public window: a wider window means fewer flickers but later switches.

In [ ]:
pd.concat([show(f"organisers: {n}", pm.read_label_csv(f"evaluation/results/baseline_{n}.csv").label.values, keep=False)
           for n in ("raw", "smooth50", "smooth125")])

## 2. Ball lost → keep the last team

Our starting point is `smooth50` (v0). The ball is missing on ~41% of graded frames; calling all of them dead is
wrong whenever play goes on without the ball being tracked. Instead, **the last team keeps the ball** until the votes
say otherwise (v1).

In [ ]:
pd.concat([show("v0: nearest team, majority over ±2 s", B.V["v0"]),
           show("v1: + ball lost -> keep the last team", B.V["v1"])])

## 3. Dead ball when the players slow down

When play stops, everyone walks. **Outfield players' mean speed** (smoothed over 3 s) **below 1.3 m/s for at least
4 s** → dead ball.

In [ ]:
show("v2: + players slow -> dead ball", B.V["v2"])

## 4. Dead ball when the ball leaves the pitch

The tracked ball **outside the pitch**, padded by 1 s on each side → dead ball.

In [ ]:
show("v3: + ball off the pitch -> dead ball", B.V["v3"])

## 5. Dead ball just before a set piece

A **SET PIECES** tag (throw-in, corner, free kick, goal kick, kick-off) means play was stopped just before it: the
**5 s before** each tag → dead ball.

In [ ]:
show("v4: + 5 s before a set-piece tag -> dead ball", B.V["v4"])

## 6. No votes while the ball is in flight

A long ball passes over players of both teams. If they vote, it is handed to whoever is underneath. A ball **free of
players (nobody within 2 m) for at least 0.3 s and reaching 8 m/s** is in flight: **nobody votes** until a player
touches it, and the last team keeps it meanwhile.

In [ ]:
show("v5: + no votes while the ball is in flight", B.V["v5"])

## 7. Anti-flicker: short suspicious possessions

The metric punishes a fake short possession far more than a late switch. A possession **shorter than 3 s, with less
than 1 s of continuous control and no completed pass** is suspicious. Two ways to remove it:
- **merge** it into the neighbouring team (v7);
- write it **`X`** (v8): the scorer fills `X` with our previous label, so the interrupted possession simply goes on.

`X` won and is kept.

In [ ]:
pd.concat([show("v7: + suspicious possessions merged", B.V["v7"], keep=False),
           show("v8: + suspicious possessions -> X", B.V["v8"])])

## 8. The referee's whistle

The audio of the tactical video is analysed frame by frame (`analysis/whistle.py`). Whistle candidates are tonal peaks
in the **3.5–4.6 kHz** band; their signature (pitch, stability, second harmonic, duration) is **learned from blows the
pipeline locates by itself**: kick-offs, ends of halves, fouls before tagged free kicks. A matching blow starts a dead
ball at its first frame, **only if the players slow down within 6 s** (whistles that restart play, or sound during a
stoppage, change nothing).

In [ ]:
print(f"refined referee blows over the match: {len(B.REF)}")
V8W = best.v8_whistle(B)
show("v8 + whistle", V8W)

## 9. Finishing touches (the submitted model)

Three small rules. **None adds or removes a possession**: each moves a boundary to a better place, all outside the
public window (so the public check cannot see them; they were accepted on full-match checks and reasoning).
- **restart end:** at a restart announced by a *start* whistle and confirmed by a set-piece restart within 3 s, the
  dead ball runs up to the restart kick and not past it (the slow-down signal lags the kick by 1–3 s);
- **tag-confirmed long passes:** a kick flying at least 1 s and 15 m inside a **LONG BALL** tag of the kicking team:
  that team keeps the ball until the reception;
- **blackout split:** the ball disappears with one team and reappears with the other while play stays live: the switch
  goes to the **middle** of the blackout instead of near its end.

In [ ]:
add, remove = best.restart_end_adjustments(B)
ing = {**B.ING, "whistle_dead": wh.dead_from_whistles(ctx, B.REF.start.values, until="slow"),
       "dead_add": add, "dead_remove": remove}
singles = {
    "v8 + whistle + restart end": v9.build(ctx, ing, **{**best.OFF, "x_policy": True, "whistle": True}),
    "v8 + whistle + tag-confirmed long passes": ps.pass_addon(V8W, best.tagged_long_passes(B)),
    "v8 + whistle + blackout split": best.blackout_split(ctx, V8W),
}
FINAL = best.final(B)
T9 = pd.concat([show(k, v, keep=False) for k, v in singles.items()] + [show("FINAL: all three (submitted)", FINAL)])
base = filled(V8W)
T9["frames moved vs v8 + whistle"] = [int((filled(v) != base)[g].sum()) for v in [*singles.values(), FINAL]]
T9

## 10. Every step in one table

In [ ]:
pd.concat([show(k, v, keep=False) for k, v in STEPS.items()])

## 11. The submission file

Written exactly as `scripts/make_submission.py` writes it, checked with the official validator's rules, and compared
with the md5 of the file we submitted.

In [ ]:
EXPECTED_MD5 = "19c22efab8064c4384e32268eefd680a"
path = "evaluation/results/pecos.csv"
pd.DataFrame({"frame": np.arange(len(FINAL)), "label": FINAL}).to_csv(path, index=False)
reason = pm.validate_submission(pm.read_label_csv(path), ctx.N)
md5 = hashlib.md5(open(path, "rb").read()).hexdigest()
print(f"{path}: {'valid' if not reason else 'INVALID: ' + reason}, {len(FINAL):,} rows")
print(f"md5 {md5} -> {'identical to the submitted file' if md5 == EXPECTED_MD5 else 'DIFFERENT from the submitted file'}")
print(pd.Series(FINAL).value_counts().reindex(list("HADX")).to_dict())